In [1]:
from dfft.utils.files import get_path
import os
import pandas as pd
import numpy as np
import re
import json

In [21]:
def extract_time(d):
    m = re.match(r"(\d{8}_\d{6})", os.path.basename(d))
    return m.group(1) if m else ""

def get_lastest_exp(base_dir):
    exp_dirs = [os.path.join(base_dir, d) for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))]
    exp_dirs = sorted(exp_dirs, key=extract_time, reverse=True)
    return exp_dirs[-1]

In [22]:
names = ['moment-small', 'moment-base', 'moment-large']
configs=["-finetuned", "-lora"]
losses = ["", "-decision"]
models = [ name + c + l for name in names for c in configs for l in losses]
models.append('moment-small-zeroshot')
models.append('moment-base-zeroshot')
models.append('moment-large-zeroshot')

In [23]:
all = []
for model in models:
    base_dir = get_path(f"results\\experiments\\{model}\\NEMA")
    exp_dir = get_lastest_exp(base_dir)
    df = pd.read_parquet(exp_dir + "/results.parquet")
    #Compute regret
    result = df.groupby('split')[['true_profit', 'real_profit']].sum()
    result['model'] = model
    result['regret'] = (result['true_profit'] - result['real_profit']) / result['true_profit']
    all.append(result.reset_index())

regret = pd.concat(all, axis= 0)
#Reshape data
regret = regret[['model', 'split', 'regret']]
regret = regret.pivot(index='model', columns='split', values='regret')
regret = regret.sort_values('val')
print(regret)

split                                test     train       val
model                                                        
moment-small-lora                0.499796  0.347621  0.521717
moment-small-finetuned           0.505636  0.344039  0.524140
moment-small-finetuned-decision  0.558967  0.525741  0.529649
moment-base-lora                 0.497311  0.314989  0.533978
moment-large-lora                0.494282  0.322596  0.534899
moment-small-lora-decision       0.523186  0.519723  0.535227
moment-base-finetuned            0.493745  0.322926  0.535583
moment-base-finetuned-decision   0.537795  0.498197  0.539099
moment-large-lora-decision       0.537646  0.480710  0.543122
moment-large-finetuned           0.491269  0.317369  0.543864
moment-large-finetuned-decision  0.536896  0.521566  0.551329
moment-base-lora-decision        0.526396  0.513401  0.556094
moment-large-zeroshot            0.955504  1.093350  0.974166
moment-small-zeroshot            1.044120  1.504422  1.058501
moment-b

In [24]:
all=[]
for model in models:
    base_dir = get_path(f"results\\experiments\\{model}\\NEMA")
    exp_dir = get_lastest_exp(base_dir)
    with open(exp_dir + "/results.json", 'r') as f:
            data = json.load(f)
    df = pd.DataFrame([model], columns=["model"])
    df["test_mse"] = data['test_results']['mse']
    df["val_mse"] = data['final_val_results']['mse']
    df["train_mse"] = data['final_train_results']['mse']
    all.append(df)

mse  = pd.concat(all, axis= 0)
print(mse)

                             model     test_mse      val_mse    train_mse
0           moment-small-finetuned  1553.311315  1275.799409  1474.608034
0  moment-small-finetuned-decision  2168.204164  1549.871845  2535.893687
0                moment-small-lora  1548.790662  1277.224637  1487.036731
0       moment-small-lora-decision  2298.288294  1628.013287  2689.121959
0            moment-base-finetuned  1569.585921  1282.859550  1410.832785
0   moment-base-finetuned-decision  2113.800044  1510.454729  2482.394454
0                 moment-base-lora  1570.589465  1280.933352  1396.941941
0        moment-base-lora-decision  2147.396678  1529.401890  2521.618161
0           moment-large-finetuned  1543.557585  1288.532773  1454.693332
0  moment-large-finetuned-decision  2119.466319  1506.603870  2476.742357
0                moment-large-lora  1540.555288  1287.977409  1435.890717
0       moment-large-lora-decision  2110.937691  1514.112825  2475.080531
0            moment-small-zeroshot  22

In [25]:
metrics = pd.merge(mse, regret, on='model', how='inner')
print(metrics.sort_values('val'))

                              model     test_mse      val_mse    train_mse  \
2                 moment-small-lora  1548.790662  1277.224637  1487.036731   
0            moment-small-finetuned  1553.311315  1275.799409  1474.608034   
1   moment-small-finetuned-decision  2168.204164  1549.871845  2535.893687   
6                  moment-base-lora  1570.589465  1280.933352  1396.941941   
10                moment-large-lora  1540.555288  1287.977409  1435.890717   
3        moment-small-lora-decision  2298.288294  1628.013287  2689.121959   
4             moment-base-finetuned  1569.585921  1282.859550  1410.832785   
5    moment-base-finetuned-decision  2113.800044  1510.454729  2482.394454   
11       moment-large-lora-decision  2110.937691  1514.112825  2475.080531   
8            moment-large-finetuned  1543.557585  1288.532773  1454.693332   
9   moment-large-finetuned-decision  2119.466319  1506.603870  2476.742357   
7         moment-base-lora-decision  2147.396678  1529.401890  2